# DS2002 · Midterm Progress

**Lab — 2026-10-30 · Fall 2026**  

---

## Lab 10 — Midterm progress, week 2 of 3

One submission per team. Questions 1 through 3 should be substantially answered by now — not polished, but answered, with a number and a chart each.

Presentations are Monday. If a question is not working, this is the last checkpoint where there is time to do something about it, so report the actual state rather than the intended one.

### Q1 — Demand surge

**TODO:** your baseline window, your surge window, the average for each, and the multiple. Plus the chart. State the window boundaries explicitly — a multiple without a window is not a finding.

In [3]:
import seaborn as sns
import pandas as pd

DECISIONS = []
def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def better_consolidate(df: pd.DataFrame, mapping: dict, column: str, target_column: str) -> pd.DataFrame:
    out = df.copy()
    # .map() replaces values found in the dictionary, leaving unmapped values as NaN,
    # then .fillna() puts the original column value back if no mapping exists.
    out[target_column] = out[column].map(mapping).fillna(out[column])
    return out

def remove_duplicates(df):
    df = df.copy()
    after = df.drop_duplicates()
    removed = len(df) - len(after)
    log('duplicates', 'dropped exact duplicate rows', removed)
    return after


def clean_unit_price(df, column ='unit_price'):
    before = df.copy()
    df = df.copy()
    df[column] = df[column].astype(str).str.replace('$', '', regex=False).str.strip()
    df[column] = pd.to_numeric(df[column], errors='coerce')
    log('clean unit prices', 'strip and to numeric', len(df.compare(before)))
    return df

def neg_quant(df, column = 'quantity'):
    before = df.copy()
    df = df.copy()
    if column in df.columns:
        df = df[df[column]>=0]
    removed = len(before) - len(df)
    log('negative quantity', 'drop negative quantities', removed)
    return df

def clean_store_id(df, column = 'store_id'):
    before = df.copy()
    df = df.copy()
    if column in df.columns:
        df[column] = df[column] = df[column].astype(str).str.strip().str.upper()
    log('clean store id', 'strip and upper', len(df.compare(before)))
    return df


def parse_timestamp(df, column = 'timestamp'):
    df = df.copy()
    rows_parsed = 0
    unparsed_count = 0
    if column in df.columns:
        values = df[column].astype('string').str.strip()
        parsed = pd.to_datetime(values, format='mixed', errors='coerce')
        unparsed_count = int((values.notna() & values.ne('') & parsed.isna()).sum())
        df[column] = parsed
        rows_parsed = int(parsed.notna().sum())
    log(
        'parse timestamps',
        f'parsed mixed formats; {unparsed_count} non-empty value(s) were unparseable',
        rows_parsed,
    )
    return df

def lowercase_category_names(df, column='category'):
    df = df.copy()
    rows_changed = 0
    if column in df.columns:
        before = df[column]
        df[column] = before.str.lower()
        rows_changed = int((before.ne(df[column]) & before.notna()).sum())
    log('lowercase category names', 'convert category names to lowercase', rows_changed)
    return df

def standardize_category_names(df, column='category'):
    mappings = {
        'emergency supplies': 'emergency',
        'breakfast foods': 'breakfast',
        'groceries': 'grocery',
        'household essentials': 'household',
        'baby products': 'baby',
        'alcoholic beverages': 'alcohol',
        'health & wellness': 'health',
        'games & entertainment': 'entertainment'
    }
    df = df.copy()
    rows_changed = 0
    if column in df.columns:
        before = df[column]
        df = better_consolidate(df, mappings, "category", "category")
        rows_changed = int((before.ne(df[column]) & before.notna()).sum())
    log('consolidate category names', 'consolidate some categories that meant the same thing', rows_changed)
    return df

DECISIONS = []

path = '../data/walmart_transactions.csv'
df = pd.read_csv(path)

df = remove_duplicates(df)
df = clean_unit_price(df)
df = neg_quant(df)
df = clean_store_id(df)
df = lowercase_category_names(df)
df = parse_timestamp(df)
df = standardize_category_names(df)

def show_log():
    return pd.DataFrame(DECISIONS)

print(df.shape)
df['date'] = df['timestamp'].dt.normalize()

show_log()

[duplicates] dropped exact duplicate rows (30 row(s))
[clean unit prices] strip and to numeric (25532 row(s))
[negative quantity] drop negative quantities (130 row(s))
[clean store id] strip and upper (1785 row(s))
[lowercase category names] convert category names to lowercase (19020 row(s))
[parse timestamps] parsed mixed formats; 0 non-empty value(s) were unparseable (24521 row(s))
[consolidate category names] consolidate some categories that meant the same thing (5556 row(s))
(25402, 8)


,step,decision,rows
0,duplicates,dropped exact duplicate rows,30
1,clean unit prices,strip and to numeric,25532
2,negative quantity,drop negative quantities,130
3,clean store id,strip and upper,1785
4,lowercase category names,convert category names to lowercase,19020
5,parse timestamps,parsed mixed formats; 0 non-empty value(s) wer...,24521
6,consolidate category names,consolidate some categories that meant the sam...,5556


In [9]:
category_count = df.groupby(["date", "category"]).agg(
    count=('category', 'count')
).reset_index()

unique_dates = sorted(category_count['date'].unique())
first_15_dates = unique_dates[:15]

# Calculate baseline average count per category over the first 15 days
baseline_df = (
    category_count[category_count['date'].isin(first_15_dates)]
    .groupby('category')['count']
    .mean()
    .reset_index(name='baseline_avg')
)

# 7 days before hurricane
target_start = pd.to_datetime('2024-09-19')
target_end = pd.to_datetime('2024-09-25')
target_dates = pd.date_range(start=target_start, end=target_end)

# Calculate average count per category during this 7-day window
period_df = (
    category_count[category_count['date'].isin(target_dates)]
    .groupby('category')['count']
    .mean()
    .reset_index(name='week_before_landfall_avg')
)

comparison_df = pd.merge(baseline_df, period_df, on='category')
comparison_df['pct_increase'] = (
    (comparison_df['week_before_landfall_avg'] - comparison_df['baseline_avg'])
    / comparison_df['baseline_avg']
) * 100

comparison_df.sort_values(by="pct_increase", ascending=False)

cleaned_transactions = df


**Q1 answer, in one sentence a manager could read:** _The breakfast category's transactions experienced the greatest sales surge leading up to the hurricane's landfall._

### Q2 — SKU consolidation

**TODO:** how many distinct SKUs you started with, how many canonical products you ended with, your largest consolidated family, and a before-and-after comparison showing what changes. Include the assertion proving consolidation did not change unit totals.

In [1]:
from pandas import DataFrame
# TODO
import pandas as pd
def create_mappings(from_list: list[str], to: str) -> dict[str, str]:
    d = {}
    for fr in from_list:
        d[fr] = to
    return d

CANON = {}
# |= merges dictionaries
CANON |= create_mappings(["SKU#459812", "POPTART-STRAW", "PT_STRAWBERRY"],
                         "PT-12")
CANON |= create_mappings(["WTR-24", "WATER24PK", "WTR_24PK"],
                         "WTR-24")
CANON |= create_mappings(['BG-MONO', 'MONOPOLY-01', 'GAME-MONO'],
                         'BG-MONO')
CANON |= create_mappings(['BEER-24', 'LTBEER24'],
                         'BEER-24')
CANON |= create_mappings(['BATT-100', 'BAT-AA4', 'BATT_AA_4PK' ],
                         'BATT-AA')
CANON |= create_mappings(['FLASH-01','LED-FLASH'],
                         'LED-FLASH')
CANON |= create_mappings(['CARDS-01','PLAYCARD-STD'],
                         'PLAYCARD-STD')
CANON |= create_mappings(['STREAM-01','ROKU-STD'],
                         'STREAM-01')
CANON |= create_mappings(['SOUP-CAN','CNSOUP-01',],
                         'CNSOUP-01')
CANON |= create_mappings(['BREAD-WH', 'WHITEBREAD'],
                         'BREAD-24')
CANON |= create_mappings(['PB-16' , 'PNUTBTR-16'],
                         'PB-16')
CANON |= create_mappings(['CNDL-EM' ,'CANDLE-EMRG'],
                         'CANDLE-EM' )
CANON |= create_mappings(['FAID-01', 'FIRSTAID-KIT'],
                         'FAID-01' )
CANON |= create_mappings(['CHRG-PORT' , 'PORTCHARGER'],
                         'CHRG-PORT' )
CANON |= create_mappings(['CHIPS-VP', 'CHIPVARIETY'],
                         'CHIPS-VP' )
CANON |= create_mappings(['PTOWEL-6', 'PAPERTOWEL6'],
                         'PTOWEL-6')
CANON |= create_mappings(['TP-12', 'TOILETPAPER12'],
                         'TP-12')
CANON |= create_mappings(['DIAP-S3', 'DIAPERS-3'],
                         'DIAP-S3')
CANON |= create_mappings(['BFORM-01', 'BABYFORMULA'],
                         'BFORM-01')
CANON |= create_mappings(['TARP-HD', 'HDTARP-01'],
                         'TARP-HD')


In [6]:
df = pd.read_csv('../data/product_catalog.csv')

def consolidate(frame: DataFrame):
    out = frame.copy()
    out['canonical_sku'] = out.apply(apply_canonical_sku, axis=1)
    return out

def apply_canonical_sku(row):
    if row['sku'] in CANON:
        return CANON[row['sku']]
    return row['sku']

consolidated = consolidate(df)
consolidated

,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical_sku
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,PT-12
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,PT-12
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,PT-12
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack,PT-12
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack,WTR-24
5,WATER24PK,Bottled Water 24-Pack,Emergency,Bottled,2.38,Sara Lee,24 pack,WTR-24
6,WTR_24PK,Water - 24 Pack,Emergency,NaN,2.08,Frito-Lay,1 pack,WTR-24
7,BATT-100,BATTERIES AA 4PK,Emergency Supplies,AA,NaN,3M Company,6 pack,BATT-AA
8,BAT-AA4,AA Batteries 4-Pack,Emergency Supplies,AA,4.16,Kimberly-Clark,4pk,BATT-AA
9,BATT_AA_4PK,AA Batteries 4-Pack,Emergency,AA,3.71,Anheuser-Busch,24 pack,BATT-AA


In [4]:
cleaned_transactions

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN012636,FL-710,2024-09-24 09:48:13,PNUTBTR-16,PEANUT BUTTER,grocery,1,3.15
1,TXN000759,FL-412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,grocery,1,2.02
2,TXN003338,FL-105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,household,2,10.47
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,electronics,1,28.59
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,alcohol,3,18.82
...,...,...,...,...,...,...,...,...
25557,TXN020201,FL-105,2024-09-03 18:28:07,DIAPERS-3,Diapers Size 3,baby,2,24.32
25558,TXN014835,FL-239,2024-09-26 08:47:08,CARDS-01,Playing Cards,entertainment,3,5.00
25559,TXN002434,FL-239,2024-09-16 16:39:00,PAPERTOWEL6,Paper Towels 6-Roll,household,2,7.78
25560,TXN014739,FL-710,2024-09-25 17:08:35,CHRG-PORT,PORTABLE CHARGER,electronics,1,24.18


In [7]:
consolidated_transactions = consolidate(cleaned_transactions)
consolidated_transactions

,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,canonical_sku
0,TXN012636,FL-710,2024-09-24 09:48:13,PNUTBTR-16,PEANUT BUTTER,grocery,1,3.15,PB-16
1,TXN000759,FL-412,2024-09-13 20:30:20,CNSOUP-01,CANNED SOUP,grocery,1,2.02,CNSOUP-01
2,TXN003338,FL-105,2024-09-17 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,household,2,10.47,TP-12
3,TXN025020,FL-330,2024-09-11 06:29:08,ROKU-STD,Roku Streaming Stick,electronics,1,28.59,STREAM-01
4,TXN015797,FL-302,2024-09-26 15:59:42,BEER-24,Light Beer 24pk,alcohol,3,18.82,BEER-24
...,...,...,...,...,...,...,...,...,...
25557,TXN020201,FL-105,2024-09-03 18:28:07,DIAPERS-3,Diapers Size 3,baby,2,24.32,DIAP-S3
25558,TXN014835,FL-239,2024-09-26 08:47:08,CARDS-01,Playing Cards,entertainment,3,5.00,PLAYCARD-STD
25559,TXN002434,FL-239,2024-09-16 16:39:00,PAPERTOWEL6,Paper Towels 6-Roll,household,2,7.78,PTOWEL-6
25560,TXN014739,FL-710,2024-09-25 17:08:35,CHRG-PORT,PORTABLE CHARGER,electronics,1,24.18,CHRG-PORT


**Q2 answer:** _..._

### Q3 — Weather correlation

**TODO:** the API call, the validated join, and the relationship you found. Include:

- the row count before and after the join, proving nothing was lost
- whether the data is live or fallback
- how many days the correlation is computed over

If your n is small, say so here rather than being asked about it on Monday.

In [ ]:
# TODO

**Q3 answer, with its caveat:** _..._

### Q4 and Q5 — where they stand

**TODO:** one honest sentence each on the remaining two questions and who is finishing them this weekend.

**Q4 (stores and geography):** _..._

**Q5 (signal or artifact):** _..._

### Self-check

In [ ]:
progress = {
    'q1_multiple': None,            # TODO: a number
    'q1_baseline_window': 'TODO',   # TODO: stated explicitly
    'q2_canonical_products': None,  # TODO: a number
    'q3_join_rows_preserved': None, # TODO: True/False
    'q3_days_in_correlation': None, # TODO: a number
    'charts_with_message_titles': None,  # TODO: how many
    'notebook_runs_clean': False,   # TODO: True after Restart and Run All
    'repo_pushed': False,           # TODO
}

for label, value in progress.items():
    filled = not (value is None or value is False
                  or (isinstance(value, str) and value.startswith('TODO')))
    print('PASS' if filled else 'FAIL', '-', label, '=', value)
print()
print('ready for presentations:', all(
    not (v is None or v is False or (isinstance(v, str) and v.startswith('TODO')))
    for v in progress.values()))

### Write-up

**a)** One finding that surprised your team, and what you checked to make sure it was real rather than a bug.

**b)** One number you reported that you are least confident in, and why.

_your answer here_